# 2. Hafta — BPE Tokenizer Test Notebook

Bu notebook, `hafta2_tokenizer/train_bpe_tokenizer.py` ile eğitilip
[yoitsmeyusuf/felsefe-bpe-tokenizer](https://huggingface.co/yoitsmeyusuf/felsefe-bpe-tokenizer)'a
push edilen BPE tokenizer'ın **nasıl çalıştığını** örneklerle gösterir:

1. Temel bilgiler (vocab, özel tokenlar)
2. Encode / decode round-trip
3. BPE'nin byte → karakter → sık ek → kelime sırasıyla nasıl "öğrendiği"
4. Sık vs nadir kelimelerde alt-kelime (subword) parçalanması
5. Korpus genelinde karakter/token sıkıştırma oranı
6. Domain-içi (felsefe) vs domain-dışı metinde davranış farkı
7. `VOCAB_SIZE` seçiminin canlı bir deneyle doğrulanması (500 / 2000 / 8000 / 32000)


In [1]:
from pathlib import Path
from transformers import AutoTokenizer

REPO_ID = "yoitsmeyusuf/felsefe-bpe-tokenizer"
tok = AutoTokenizer.from_pretrained(REPO_ID)

# corpus.txt'yi hem proje kökünden hem de hafta2_tokenizer/ içinden çalıştırınca bulsun
_candidates = [Path("hafta2_tokenizer/data/corpus.txt"), Path("data/corpus.txt")]
CORPUS_PATH = next((p for p in _candidates if p.exists()), None)
assert CORPUS_PATH is not None, "corpus.txt bulunamadı — notebook'u proje kökünden veya hafta2_tokenizer/ içinden çalıştırın"

print(f"Tokenizer yüklendi: {REPO_ID}")
print(f"Korpus: {CORPUS_PATH.resolve()}")

./.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
./.venv/lib/python3.13/site-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Tokenizer yüklendi: yoitsmeyusuf/felsefe-bpe-tokenizer
Korpus: ./hafta2_tokenizer/data/corpus.txt


## 1. Temel bilgiler

Tokenizer, `train_bpe_tokenizer.py`'de tanımlı özel tokenlarla birlikte
byte-level bir BPE. `vocab_size` script'te 8000 olarak ayarlandı (32000
değil — nedenini bölüm 7'de canlı olarak göreceğiz).

In [2]:
print("vocab_size:", tok.vocab_size)
print("özel tokenlar:", tok.special_tokens_map)
print("toplam benzersiz vocab girişi:", len(tok.get_vocab()))

vocab_size: 8000
özel tokenlar: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}
toplam benzersiz vocab girişi: 8000


## 2. Encode / Decode round-trip

Bir cümleyi token id'lerine çevirip geri metne döndürüyoruz — byte-level
BPE olduğu için decode, orijinal metinle (boşluk/noktalama dahil) birebir
eşleşmeli.

In [3]:
sentence = "Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir."

ids = tok.encode(sentence)
tokens = tok.convert_ids_to_tokens(ids)
decoded = tok.decode(ids)

print("orijinal :", sentence)
print("token id'leri:", ids)
print("token sayısı:", len(ids), "/ kelime sayısı:", len(sentence.split()))
print()
print("parçalanma:")
for t in tokens:
    print(f"  {t!r:20s} -> {tok.convert_tokens_to_string([t])!r}")
print()
print("decode    :", decoded)
print("birebir eşleşiyor mu:", decoded == sentence)

orijinal : Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir.
token id'leri: [3215, 11, 819, 3515, 800, 5088, 520, 1560, 612, 698, 6431, 18]
token sayısı: 12 / kelime sayısı: 8

parçalanma:
  'Nietzsche'          -> 'Nietzsche'
  "'"                  -> "'"
  'nin'                -> 'nin'
  'ĠvaroluÅŁÃ§uluk'    -> ' varoluşçuluk'
  'ĠÃ¼zerine'          -> ' üzerine'
  'ĠdÃ¼ÅŁÃ¼nceleri'    -> ' düşünceleri'
  'ĠÃ¶zgÃ¼r'           -> ' özgür'
  'Ġirade'             -> ' irade'
  'Ġkavram'            -> ' kavram'
  'Ä±yla'              -> 'ıyla'
  'ĠiliÅŁkilidir'      -> ' ilişkilidir'
  '.'                  -> '.'

decode    : Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir.
birebir eşleşiyor mu: True


## 3. BPE nasıl "öğreniyor"? Byte → karakter → sık ek → kelime

Byte-level BPE, 256 ham byte'lık bir alfabeyle başlar (+ bizim 6 özel
tokenımız). Türkçe karakterler (ı, ş, ğ, ü, ö, ç) UTF-8'de 2 byte olduğu
için bunlar bile ilk birkaç "merge" (birleştirme) ile tek tokene dönüşüyor.
Ardından en sık geçen çift-harfler (ek parçaları: "in", "ir", "en"...),
en son da tam kelimeler/kökler öğreniliyor. `vocab`'taki id sırası, bu
öğrenme sırasını gösteriyor (küçük id = erken/temel, büyük id = geç/nadir).

In [4]:
vocab = tok.get_vocab()
by_id = sorted(vocab.items(), key=lambda kv: kv[1])

def readable(token):
    return tok.convert_tokens_to_string([token])

# ilk çok-karakterli (yani gerçek bir "merge" sonucu olan) token'ı bul
first_merge_id = next(i for t, i in by_id if len(t) > 1 and not t.startswith("<"))

print(f"0-5      : özel tokenlar          -> {[t for t, i in by_id[:6]]}")
print(f"6-{first_merge_id-1:<4d} : ham byte alfabesi (henüz merge yok)")
print()
print(f"{first_merge_id}-{first_merge_id+9} : İLK öğrenilen merge'ler (en temel/en sık):")
for t, i in by_id[first_merge_id:first_merge_id+10]:
    print(f"  id={i:5d}  -> {readable(t)!r}")

print()
print("7990-7999: SON öğrenilen merge'ler (en nadir, korpusa en özgü):")
for t, i in by_id[7990:8000]:
    print(f"  id={i:5d}  -> {readable(t)!r}")

0-5      : özel tokenlar          -> ['<unk>', '<pad>', '<|endoftext|>', '<|user|>', '<|assistant|>', '<|system|>']
6-126  : ham byte alfabesi (henüz merge yok)

127-136 : İLK öğrenilen merge'ler (en temel/en sık):
  id=  127  -> 'ı'
  id=  128  -> 'in'
  id=  129  -> ' b'
  id=  130  -> 'an'
  id=  131  -> 'ar'
  id=  132  -> 'ir'
  id=  133  -> 'ü'
  id=  134  -> 'er'
  id=  135  -> ' d'
  id=  136  -> 'en'

7990-7999: SON öğrenilen merge'ler (en nadir, korpusa en özgü):
  id= 7990  -> 'dıran'
  id= 7991  -> 'dırap'
  id= 7992  -> 'yaş'
  id= 7993  -> 'yahu'
  id= 7994  -> 'yaşam'
  id= 7995  -> 'yahud'
  id= 7996  -> 'yavel'
  id= 7997  -> ' saul'
  id= 7998  -> ' salak'
  id= 7999  -> ' sadak'


## 4. Sık kelime vs nadir kelime: alt-kelime parçalanması

Korpusta sık geçen felsefe terimleri/isimleri tek (veya çok az) token'a
sığmalı; hiç görülmemiş/nadir kelimeler ise daha fazla parçaya bölünmeli —
bu, BPE'nin OOV (out-of-vocabulary) kelimeleri nasıl idare ettiğini gösterir.

In [5]:
words = [
    "Nietzsche",       # korpusta çok sık geçen bir filozof adı
    "felsefe",         # domain'in ana kelimesi
    "varoluşçuluk",    # sık ama uzun/bileşik bir terim
    "özgür",
    "irade",
    "epistemoloji",    # daha az sık geçen bir terim
    "antropomorfizasyonculaştıramadıklarımızdanmışsınızcasına",  # tamamen uydurma, çok uzun bir kelime
]

for w in words:
    pieces = tok.convert_ids_to_tokens(tok.encode(w))
    readable_pieces = [readable(p) for p in pieces]
    print(f"{w!r:35s} -> {len(pieces)} parça: {readable_pieces}")

'Nietzsche'                         -> 1 parça: ['Nietzsche']
'felsefe'                           -> 1 parça: ['felsefe']
'varoluşçuluk'                      -> 2 parça: ['var', 'oluşçuluk']
'özgür'                             -> 1 parça: ['özgür']
'irade'                             -> 2 parça: ['ira', 'de']
'epistemoloji'                      -> 2 parça: ['ep', 'istemoloji']
'antropomorfizasyonculaştıramadıklarımızdanmışsınızcasına' -> 17 parça: ['ant', 'r', 'op', 'om', 'orf', 'iz', 'asyon', 'cu', 'laştır', 'ama', 'dık', 'larımız', 'dan', 'mış', 'sınız', 'c', 'asına']


## 5. Korpus genelinde karakter/token sıkıştırma oranı

Bir tokenizer'ın "verimliliği" genelde ortalama kaç karakteri tek bir
token'a sığdırabildiğiyle ölçülür (yüksek = daha sıkıştırılmış temsil).

In [6]:
with open(CORPUS_PATH, encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

sample = lines[:200]  # hız için ilk 200 satır
sample_chars = sum(len(l) for l in sample)
sample_tokens = sum(len(tok.encode(l)) for l in sample)

print(f"örneklem: {len(sample)} satır, {sample_chars} karakter")
print(f"toplam token: {sample_tokens}")
print(f"ortalama karakter/token: {sample_chars/sample_tokens:.2f}")
print(f"(karşılaştırma: ham byte-level UTF-8 kodlamada bu oran ~1.0-1.2 olurdu)")

örneklem: 200 satır, 35317 karakter
toplam token: 8084
ortalama karakter/token: 4.37
(karşılaştırma: ham byte-level UTF-8 kodlamada bu oran ~1.0-1.2 olurdu)


## 6. Domain-içi vs domain-dışı metin karşılaştırması

Bu tokenizer sadece Türkçe **felsefe** metniyle eğitildi. Aynı uzunlukta
ama tamamen farklı bir alandan (teknoloji/İngilizce karışık jargon) bir
cümleyle karşılaştırınca, tokenizer'ın domain'e ne kadar "özelleştiğini"
görürüz — domain-dışı metin çok daha fazla token'a bölünür (çoğunlukla
ham byte/karakter seviyesine geriler).

In [7]:
in_domain = "Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir."
out_domain = "Docker container'ı Kubernetes cluster'ında horizontal pod autoscaler ile ölçeklendirdik."

for label, text in [("domain-içi (felsefe)", in_domain), ("domain-dışı (teknoloji)", out_domain)]:
    ids = tok.encode(text)
    n_words = len(text.split())
    print(f"{label}:")
    print(f"  {text}")
    print(f"  {n_words} kelime -> {len(ids)} token  (token/kelime = {len(ids)/n_words:.2f})")
    print()

domain-içi (felsefe):
  Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir.
  8 kelime -> 12 token  (token/kelime = 1.50)

domain-dışı (teknoloji):
  Docker container'ı Kubernetes cluster'ında horizontal pod autoscaler ile ölçeklendirdik.
  9 kelime -> 40 token  (token/kelime = 4.44)



## 7. `VOCAB_SIZE` seçiminin etkisi: canlı mini deney

README'de 32000 yerine 8000 seçmemizin gerekçesini tartışmıştık: korpus
küçük olduğu için (~380K karakter) çok büyük bir hedef vocab'a
ulaşılamayacağını ve/veya anlamsız ezberlemeye kayacağını iddia etmiştik.
Burada aynı korpus üzerinde farklı `vocab_size` hedefleriyle küçük,
tek-kullanımlık (Hub'a push edilmeyen) tokenizer'lar eğitip bunu doğrudan
gözlemleyelim.

In [8]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

def train_mini(vocab_size: int) -> Tokenizer:
    t = Tokenizer(models.BPE(unk_token="<unk>"))
    t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    t.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["<unk>"], min_frequency=2)

    def corpus_iter():
        with open(CORPUS_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield line

    t.train_from_iterator(corpus_iter(), trainer=trainer)
    return t

test_sentence = "Nietzsche'nin varoluşçuluk üzerine düşünceleri özgür irade kavramıyla ilişkilidir."

print(f"{'hedef vocab_size':>18} | {'gerçek vocab':>13} | {'test cümlesi token sayısı':>26}")
print("-" * 65)
for vs in [500, 2000, 8000, 32000]:
    mini = train_mini(vs)
    n_ids = len(mini.encode(test_sentence).ids)
    print(f"{vs:>18} | {mini.get_vocab_size():>13} | {n_ids:>26}")

  hedef vocab_size |  gerçek vocab |  test cümlesi token sayısı
-----------------------------------------------------------------





               500 |           500 |                         40





              2000 |          2000 |                         18





              8000 |          8000 |                         12





             32000 |         10750 |                         12


**Gözlem:** `vocab_size=32000` hedeflendiğinde, trainer korpusta yeterli
sıklıkta tekrarlanan çift bulamadığı için (`min_frequency=2`) gerçek vocab
32000'e hiç ulaşamıyor — çok daha küçük bir sayıda kalıyor. Ayrıca 8000 ile
"gerçek" (daha büyük) vocab arasında test cümlesi için token sayısı
neredeyse hiç değişmiyor — yani 8000'in üstüne çıkmanın bu korpus için
somut bir faydası yok, sadece daha az sıklıkla görülen (dolayısıyla daha
az güvenilir öğrenilmiş) merge'ler ekliyor. Bu da README'deki `VOCAB_SIZE
= 8000` seçimini doğrudan doğruluyor.

## Özet

- Tokenizer, Türkçe'nin özel karakterlerini ve sık eklerini ilk birkaç
  yüz merge'de öğrenmiş, felsefe'ye özgü kelimeleri (kişi adları, kavramlar)
  tek token'a indirmiş.
- Domain-içi metinde ~1.5 token/kelime, domain-dışı metinde ~4.4 token/kelime
  — tokenizer'ın felsefe diline ne kadar özelleştiğinin somut kanıtı.
- `vocab_size=8000` seçimi, korpusun büyüklüğüyle orantılı; 32000 hedefi
  pratikte ulaşılamıyor ve fayda sağlamıyor.
